In [41]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import math

import plotly.graph_objects as go

In [42]:
def get_digits(price_step):
    return round(-math.log10(price_step))

In [43]:
TICKER          = "GC=F"
INTERVAL        = "1m"

BALANCE         = 10_000     # starting balance

CONTRACT_SIZE   = 100
LOT_SIZE        = 0.01

GAP             = 4
N_LEVELS        = 15  
    
MAX_LOSS        = 10     
PROFIT_TARGET   = 5

TICKER_PRICE_PRECISION = get_digits(0.01)  # e.g. EURUSD=X has a price step of 0.0001

### Load data


In [44]:
# df = yf.download(TICKER, period="max", interval=INTERVAL)

# df.columns.names = [None, None]

# df.columns = df.columns.get_level_values(0)

# df = df.drop(columns=["Volume"])

# ----------------------------

df = pd.read_csv("data.csv")

df.set_index("Datetime", inplace=True)

df = df[["Open", "High", "Low", "Close"]]

# ----------------------------

df

,Open,High,Low,Close
Datetime,,,,
2026-04-29 13:41:00,4518.505,4523.045,4518.505,4520.995
2026-04-29 13:42:00,4520.965,4520.965,4516.845,4519.165
2026-04-29 13:43:00,4519.065,4519.605,4513.945,4514.595
2026-04-29 13:44:00,4514.625,4517.045,4513.315,4514.495
2026-04-29 13:45:00,4514.515,4515.675,4509.875,4510.395
...,...,...,...,...
2026-08-10 11:55:00,4334.865,4335.585,4334.375,4335.485
2026-08-10 11:56:00,4335.605,4335.605,4333.955,4334.065
2026-08-10 11:57:00,4334.385,4334.735,4333.255,4334.305


### Backtester

In [45]:
# Grid max possible loss = 
class GridTrendBacktester:
    def __init__(
            self,
            n_levels=N_LEVELS,
            grid_gap=GAP,
            units_per_order=LOT_SIZE * CONTRACT_SIZE,
            profit_target=PROFIT_TARGET,
            loss_limit=MAX_LOSS,
            balance=BALANCE,
    ):
        self.n_levels = n_levels
        self.grid_gap = grid_gap
        self.units_per_order = units_per_order
        self.profit_target = profit_target
        self.loss_limit = loss_limit
        self.balance = balance
        self.resets = 0

    def _build_grid(self, center_price):
        self.resets += 1
        buy_stops = {round(center_price + i * self.grid_gap, TICKER_PRICE_PRECISION): False for i in range(1, self.n_levels + 1)}
        sell_stops = {round(center_price - i * self.grid_gap, TICKER_PRICE_PRECISION): False for i in range(1, self.n_levels + 1)}
        return buy_stops, sell_stops

    def max_possible_loss(self):
        return  self.units_per_order * self.grid_gap * (self.n_levels * (self.n_levels + 1) / 2)

    def _PnL(self, price, buy_stops, sell_stops):
        active_buys = [price for price, active in buy_stops.items() if active]
        active_sells = [price for price, active in sell_stops.items() if active]

        floating_buys = np.sum((price - active_buys) * self.units_per_order)
        floating_sells = np.sum((active_sells - price) * self.units_per_order)
        return floating_buys + floating_sells
    
    def run(self, df):
        self.loss_limit = min(self.loss_limit, self.max_possible_loss())

        high = df["High"].to_numpy()
        low = df["Low"].to_numpy()
        close = df["Close"].to_numpy()

        floating_equity = np.array([])
        realized_equity = np.array([])

        buy_stops, sell_stops = self._build_grid(close[0])

        for i in range(0, len(df)):
            # Check for buy stop triggers
            for buy_order in [price for price, active in buy_stops.items() if not active]:
                if high[i] > buy_order:
                    buy_stops[buy_order] = True

            # Check for sell stop triggers
            for sell_order in [price for price, active in sell_stops.items() if not active]:
                if low[i] < sell_order:
                    sell_stops[sell_order] = True

            # ----------------------------

            current_price = close[i]

            # Handle floating equity and check for profit target or loss limit
            pnl = self._PnL(current_price, buy_stops, sell_stops)
            floating_equity = np.append(floating_equity, pnl)

            if pnl >= self.profit_target or pnl <= -self.loss_limit:
                realized_equity = np.append(realized_equity, pnl)
                # restart grid
                buy_stops, sell_stops = self._build_grid(current_price)
                continue
            else:
                realized_equity = np.append(realized_equity, 0)
    
        # balance = self.balance + np.cumsum(realized_equity)
        # equity = self.balance + np.cumsum(floating_equity)

        balance = self.balance + np.cumsum(realized_equity)
        equity = balance + floating_equity

        return pd.DataFrame({ "balance": balance, "equity": equity }, index=df.index)

### Run a backtest

Tune these to taste. `grid_gap` and `profit_target`/`loss_limit` should be sized
together with `units_per_order` — e.g. with `units_per_order=10,000` (0.1 lot), a
1-pip (0.0001) move on one filled level is worth $1.


In [46]:
bt = GridTrendBacktester()

# bt.max_possible_loss()

backtest_df = bt.run(df)

backtest_df

,balance,equity
Datetime,,
2026-04-29 13:41:00,10000.000,10000.000
2026-04-29 13:42:00,10000.000,9997.835
2026-04-29 13:43:00,10000.000,10002.405
2026-04-29 13:44:00,10000.000,10002.505
2026-04-29 13:45:00,10009.210,10018.420
...,...,...
2026-08-10 11:55:00,12239.728,12235.703
2026-08-10 11:56:00,12239.728,12237.123
2026-08-10 11:57:00,12239.728,12236.883


### Stats

In [47]:
def backtest_stats(df):
    df = df.copy()

    # --------------------------------------------------
    # 1. Return %
    # --------------------------------------------------
    initial_balance = df["balance"].iloc[0]

    final_balance = df["balance"].iloc[-1]
    net_profit = final_balance - initial_balance
    return_pct = net_profit / initial_balance * 100

    # --------------------------------------------------
    # 2. Equity Drawdown
    # --------------------------------------------------
    equity_peak = df["equity"].cummax()

    equity_dd = equity_peak - df["equity"]
    equity_dd_pct = equity_dd / equity_peak * 100

    max_equity_dd = equity_dd.max()
    max_equity_dd_pct = equity_dd_pct.max()

    # --------------------------------------------------
    # 3. Balance Drawdown
    # --------------------------------------------------
    balance_peak = df["balance"].cummax()

    balance_dd = balance_peak - df["balance"]
    balance_dd_pct = balance_dd / balance_peak * 100

    max_balance_dd = balance_dd.max()
    max_balance_dd_pct = balance_dd_pct.max()


    # --------------------------------------------------
    # Return results
    # --------------------------------------------------
    return {
        "Initial Balance": initial_balance,
        "Final Balance": final_balance,
        "Net Profit": net_profit,
        "Return %": return_pct,

        "Max Equity DD": max_equity_dd,
        "Max Equity DD %": max_equity_dd_pct,

        "Max Balance DD": max_balance_dd,
        "Max Balance DD %": max_balance_dd_pct,
    }

def print_stats(stats):
    for name, value in stats.items():
        print(f"{name:20s}: {value:.4f}")


In [48]:
print_stats(backtest_stats(backtest_df))

Initial Balance     : 10000.0000
Final Balance       : 12239.7280
Net Profit          : 2239.7280
Return %            : 22.3973
Max Equity DD       : 537.5700
Max Equity DD %     : 4.4027
Max Balance DD      : 394.8760
Max Balance DD %    : 3.9118


### Plots

In [49]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=backtest_df.index,
    y=backtest_df["balance"],
    mode="lines",
    name="Balance"
))

fig.add_trace(go.Scatter(
    x=backtest_df.index,
    y=backtest_df["equity"],
    mode="lines",
    name="Equity"
))

fig.update_layout(
    title="Balance vs Equity",
    xaxis_title="Time",
    yaxis_title="Account Value",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

### 7. (Optional) Parameter sweep

Quick grid search over `grid_gap` / `profit_target` / `loss_limit` to see which
combinations held up over this window. This is purely descriptive of the sample
you fetched — not a substitute for out-of-sample testing or walk-forward validation.


In [50]:
# from itertools import product

# grid_gaps = [2, 3, 4, 5]
# # grid_gaps = [0 + i * 0.05 for i in range(int((1 - 0) / 0.05) + 1)]
# profit_targets = [5]
# loss_limits = [10]
# n_levels = [15]

# results = {}

# for gap, pt, ll, nl in product(grid_gaps, profit_targets, loss_limits, n_levels):
#     backtest = GridTrendBacktester(n_levels=nl, grid_gap=gap, profit_target=pt, loss_limit=ll)
#     results[(gap, pt, ll, nl)] = backtest_stats(backtest.run(df))

# top_3 = sorted(
#     results.items(),
#     key=lambda x: x[1]["Max Equity DD %"]
# )[:3]

# # only three best ones
# for (gap, pt, ll, nl), backtest_result in top_3:
#     phrase = f"Grid Gap: {gap}, Profit Target: {pt}, Loss Limit: {ll}, N Levels: {nl}"
#     print(phrase)
#     print_stats(backtest_result)
#     print("\n\n--------------------------------------------------\n\n")

# print("Done!")

## Notes / things worth tightening before trusting this

- **Fills are simplified**: a stop is assumed to fill exactly at its price whenever
  bar high/low crosses it. Real fills have slippage, especially in fast breakouts —
  the `spread` param is a blunt proxy; consider a slippage model tied to bar range.
- **No overnight/weekend handling** beyond the optional session filter — a grid left
  open across a weekend gap can take a large hit at the open.
- **1-minute Yahoo data is short-lookback and can have gaps/thin liquidity** — treat
  this as a prototype, not a robustness test. For real validation, use a broker's
  tick or 1-min history with realistic spread/slippage.
- **`replenish=True`** turns this into an unbounded-exposure trend rider (each new
  level extends the grid) — cap `n_levels` or add a max-position guard if you use it.
